# Logit Formation Analysis

How are final logits assembled? Logit buildup trajectory,
top contributors, token competition, and embedding bias.

## Why This Matters

Logit formation analyzes how the model's output predictions form and change across layers. Understanding logit formation — which components contribute what to the final prediction — is central to explaining model behavior.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Belrose et al. (2023) "Eliciting Latent Predictions"](https://arxiv.org/abs/2303.08112) — Learned affine probes improve on the logit lens

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_formation_analysis import (
    logit_buildup_trajectory, top_logit_contributors,
    logit_competition_analysis, embedding_logit_bias,
    logit_formation_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Buildup Trajectory

Track how a specific token's logit accumulates through layers.

In [ ]:
result = logit_buildup_trajectory(model, tokens, position=-1, token_id=15)
print(f"Final logit for token 15: {result['final_logit']:.4f}")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: logit={p['logit']:.4f}, "
          f"attn={p['attn_contribution']:+.4f}, mlp={p['mlp_contribution']:+.4f}")

## Top Logit Contributors

Which components contribute most to the winning token?

In [ ]:
result = top_logit_contributors(model, tokens, position=-1, top_k=8)
print(f"Top token: {result['top_token']}")
for c in result['top_contributors']:
    print(f"  L{c['layer']} {c['component']}: {c['contribution']:+.4f}")

## Logit Competition

How do top-k tokens compete for highest logit through layers?

In [ ]:
result = logit_competition_analysis(model, tokens, position=-1, top_k=3)
print(f"Tracked: {result['tracked_tokens']}")
print(f"Leader changes: {result['leader_changes']}")
for p in result['per_layer']:
    logits_str = ', '.join(f'T{t}={l:.3f}' for t, l in p['token_logits'].items())
    print(f"  Layer {p['layer']}: leader=T{p['leader']} | {logits_str}")

## Embedding Logit Bias

What predictions does the embedding layer alone suggest?

In [ ]:
result = embedding_logit_bias(model, tokens, position=-1, top_k=5)
print(f"Embed norm: {result['embed_norm']:.4f}")
print(f"Pos embed norm: {result['pos_embed_norm']:.4f}")
print("Top predictions from embedding alone:")
for tok, logit in result['top_predictions']:
    print(f"  Token {tok}: logit={logit:.4f}")

## Logit Formation Summary

In [ ]:
result = logit_formation_summary(model, tokens, position=-1)
for k, v in result.items():
    print(f"  {k}: {v}")